# 01 — Late-delivery EDA, champion evaluation and SHAP

Data: `data/gold/ml/late_delivery_training.parquet` (gold → ML contract) and the seller snapshots
`data/gold/ml/seller_features_daily.parquet` behind the Feast `seller_stats` view. Model: the
`late_delivery@champion` registry version on the MLflow server (`MLFLOW_TRACKING_URI`, default
`http://127.0.0.1:5000`).

Run top-to-bottom from anywhere inside the repo, e.g.
`uv run --project ml --with jupyter --with nbconvert --with matplotlib jupyter nbconvert --execute --to notebook --inplace ds/notebooks/01_late_delivery_eda.ipynb`.
Outputs are cleared before commit; conclusions live in `ds/reports/` and `ds/model_cards/`.
Segment tables are written to `ds/reports/tables/`, the SHAP figure to `ds/reports/figures/`.

In [ ]:
import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
os.chdir(ROOT)
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
# the server advertises presigned MinIO downloads whose host (minio:9000) only resolves inside
# Compose; from the host, stream artifacts through the tracking server instead
os.environ.setdefault("MLFLOW_ENABLE_PROXY_MULTIPART_DOWNLOAD", "false")

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import shap
from mlflow import MlflowClient
from retail_ml.config import load_late_delivery_config
from retail_ml.data import read_training
from retail_ml.features import CATEGORICAL, SELLER_FEATURES, order_features
from retail_ml.late_delivery.evaluate import classification_metrics
from retail_ml.late_delivery.train import LABEL, model_inputs, split
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
FIG = ROOT / "ds/reports/figures"
TAB = ROOT / "ds/reports/tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
BUDGET = 0.10  # alert budget: the top 10 % of risk scores in a window are flagged
cfg = load_late_delivery_config()
client = MlflowClient()

## 1. Label coverage and label rate over time
`is_late` is only defined for delivered orders. Coverage = labelled share of approved orders.

In [ ]:
raw = pd.read_parquet(cfg.training_path)
month = raw["order_approved_ts_utc"].dt.tz_localize(None).dt.to_period("M")
by_month = raw.groupby(month).agg(
    orders=("order_id", "size"),
    labelled_share=("is_late", lambda s: s.notna().mean()),
    late_rate=("is_late", lambda s: s.dropna().astype(bool).mean()),
)
print(raw["order_status"].value_counts().to_string())
by_month.tail(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
by_month["late_rate"].plot(ax=ax, marker="o")
for d in (cfg.validation_start, cfg.test_start):
    ax.axvline(pd.Period(d.tz_localize(None), "M"), color="grey", ls="--")
ax.set_title("Late rate by approval month (dashed: validation / test start)")
ax.set_ylabel("late rate")
plt.show()

## 2. Labelled set, point-in-time seller features, split balance
Seller features are joined as of each order's approval with the same semantics as the Feast
`seller_stats` view (latest snapshot with `feature_ts <= approval`, ttl 365 d, ties broken by
`created_ts`). `merge_asof` is used instead of the Feast file store because the committed host
registry points at the container path `/data/gold`; section 5 checks the join by reproducing the
champion's logged MLflow metrics.

In [ ]:
df = read_training(cfg.training_path)
snap = pd.read_parquet(ROOT / "data/gold/ml/seller_features_daily.parquet")
snap = snap.sort_values(["feature_ts", "created_ts"]).drop_duplicates(
    ["seller_id", "feature_ts"], keep="last"
)
snap["feature_ts"] = snap["feature_ts"].astype("datetime64[us, UTC]")
ent = df[["order_id", "seller_id", "order_approved_ts_utc"]].copy()
ent["ts"] = ent["order_approved_ts_utc"].astype("datetime64[us, UTC]")
joined = pd.merge_asof(
    ent.sort_values("ts"),
    snap,
    left_on="ts",
    right_on="feature_ts",
    by="seller_id",
    direction="backward",
    tolerance=pd.Timedelta(days=365),
)
joined = joined.drop_duplicates("order_id").set_index("order_id").reindex(df["order_id"])
df = pd.concat([df, joined[SELLER_FEATURES].reset_index(drop=True)], axis=1)
feats = order_features(df)
df = df.join(feats[["distance_km", "freight_ratio", "estimated_days", "approve_month"]])
parts = split(df, cfg)
balance = pd.DataFrame(
    {
        name: {
            "n": len(p),
            "late": int(p[LABEL].sum()),
            "late_rate": p[LABEL].mean(),
            "seller_feature_missing": p["seller_orders_90d"].isna().mean(),
        }
        for name, p in parts.items()
    }
).T
balance

## 3. Lateness by state, category, season (development data = train + validation windows)
The test window is kept out of the EDA; it is only used in sections 5-7.

In [ ]:
dev = pd.concat([parts["train"], parts["val"]], ignore_index=True)


def rate_table(frame, key, top=10):
    t = frame.groupby(key, observed=True)[LABEL].agg(n="size", late_rate="mean")
    t["share_of_orders"] = t["n"] / len(frame)
    return t.sort_values("n", ascending=False).head(top)


print("overall dev late rate", round(dev[LABEL].mean(), 4))
rate_table(dev, "customer_state")

In [ ]:
rate_table(dev, "product_category")

In [ ]:
dev_month = dev["order_approved_ts_utc"].dt.month
dev_dow = dev["order_approved_ts_utc"].dt.dayofweek
season = pd.concat(
    {
        "by_calendar_month": dev.groupby(dev_month)[LABEL].mean(),
        "n": dev.groupby(dev_month)[LABEL].size(),
    },
    axis=1,
)
print(season.to_string())
print("by approval weekday (0=Mon)", dev.groupby(dev_dow)[LABEL].mean().round(4).to_dict())

## 4. Distance, freight and seller effects

In [ ]:
DIST_BINS = [0, 100, 300, 600, 1000, 2000, np.inf]
FREIGHT_BINS = [0, 0.1, 0.2, 0.3, 0.5, 1.0, np.inf]
for frame in (dev, *parts.values()):
    frame["distance_bucket"] = pd.cut(frame["distance_km"], DIST_BINS, right=False)
    frame["freight_ratio_bucket"] = pd.cut(frame["freight_ratio"], FREIGHT_BINS, right=False)
    frame["same_state"] = frame["customer_state"] == frame["seller_state"]
print(dev.groupby("distance_bucket", observed=True)[LABEL].agg(n="size", late_rate="mean"))
print(dev.groupby("freight_ratio_bucket", observed=True)[LABEL].agg(n="size", late_rate="mean"))
print(dev.groupby("same_state")[LABEL].agg(n="size", late_rate="mean"))

In [ ]:
per_seller = (
    dev.groupby("seller_id")[LABEL]
    .agg(orders="size", late="sum")
    .sort_values("orders", ascending=False)
)
n_sellers = len(per_seller)
conc = {}
for q in (0.01, 0.10):
    top = per_seller.head(max(1, int(round(q * n_sellers))))
    conc[f"top_{int(q * 100)}pct_sellers"] = {
        "sellers": len(top),
        "share_of_orders": top["orders"].sum() / per_seller["orders"].sum(),
        "share_of_late": top["late"].sum() / per_seller["late"].sum(),
    }
print("sellers in dev:", n_sellers)
print(pd.DataFrame(conc).T)
hist_bucket = pd.cut(dev["seller_late_rate_90d"], [0, 1e-9, 0.05, 0.1, 0.2, 1.01], right=False)
hist_bucket = hist_bucket.astype(str).where(dev["seller_late_rate_90d"].notna(), "no history")
print("late rate by the seller's trailing-90d late rate at approval")
print(dev.groupby(hist_bucket)[LABEL].agg(n="size", late_rate="mean"))

## 5. Champion vs baselines: reproduce MLflow metrics, alert budget, calibration

In [ ]:
version = client.get_model_version_by_alias(cfg.registered_model, "champion")
champion_run = client.get_run(version.run_id)
exp = client.get_experiment_by_name(cfg.experiment)
runs = {
    r.info.run_name: r
    for r in client.search_runs([exp.experiment_id], order_by=["start_time ASC"])
    if r.data.tags.get("data_sha256") == champion_run.data.tags["data_sha256"]
}
# MLflow 3 logs models as LoggedModels: load by model id (`runs:/<id>/model` is not a run artefact)
logged_models = {
    m.source_run_id: m.model_id
    for m in mlflow.search_logged_models(experiment_ids=[exp.experiment_id], output_format="list")
}
models = {
    name: mlflow.pyfunc.load_model(f"models:/{logged_models[r.info.run_id]}")
    for name, r in runs.items()
}
print("champion version", version.version, "run", version.run_id, champion_run.info.run_name)
X = {s: model_inputs(parts[s]) for s in ("val", "test")}
y = {s: parts[s][LABEL].astype(int).to_numpy() for s in ("val", "test")}
scores = {(m, s): np.asarray(models[m].predict(X[s]), dtype=float) for m in models for s in X}
rows = []
for (m, s), p in scores.items():
    ours = classification_metrics(y[s], p)
    for k, v in ours.items():
        logged = runs[m].data.metrics[f"{s}_{k}"]
        rows.append({"model": m, "split": s, "metric": k, "recomputed": v, "mlflow": logged})
check = pd.DataFrame(rows)
# the merge_asof join differs from Feast's on a handful of rows: metrics agree to < 1e-4
print(
    "max |recomputed - mlflow|:",
    check.assign(d=(check["recomputed"] - check["mlflow"]).abs())
    .groupby(["model", "metric"])["d"]
    .max()
    .max(),
)
assert np.allclose(check["recomputed"], check["mlflow"], atol=5e-4), check
check.pivot_table(index=["model", "split"], columns="metric", values="mlflow")

In [ ]:
def at_budget(y_true, p, threshold):
    flag = p >= threshold
    tp = int((flag & (y_true == 1)).sum())
    return {
        "flagged_share": flag.mean(),
        "recall": tp / max(1, int(y_true.sum())),
        "precision": tp / max(1, int(flag.sum())),
        "lift": (tp / max(1, int(flag.sum()))) / y_true.mean(),
    }


budget_rows = []
for m in ("logistic", "lightgbm"):
    t_val = np.quantile(scores[(m, "val")], 1 - BUDGET)
    for s in ("val", "test"):
        p = scores[(m, s)]
        t_own = np.quantile(p, 1 - BUDGET)
        budget_rows.append(
            {
                "model": m,
                "split": s,
                "threshold": "own-window top 10 %",
                **at_budget(y[s], p, t_own),
            }
        )
        if s == "test":
            budget_rows.append(
                {
                    "model": m,
                    "split": s,
                    "threshold": "val top-10 % cutoff",
                    **at_budget(y[s], p, t_val),
                }
            )
budget = pd.DataFrame(budget_rows)
budget.to_csv(TAB / "late_delivery_alert_budget.csv", index=False)
print("constant prior: all scores tie, a 10 % budget is a random draw (recall 0.10, lift 1.0)")
budget

In [ ]:
calib = pd.DataFrame(
    {(m, s): {"mean_pred": scores[(m, s)].mean(), "observed": y[s].mean()} for (m, s) in scores}
).T
print("train late rate", round(parts["train"][LABEL].mean(), 4))
print(calib)
p_test = scores[("lightgbm", "test")]
dec = pd.qcut(p_test, 10, labels=False, duplicates="drop")
reliability = (
    pd.DataFrame({"p": p_test, "y": y["test"], "decile": dec})
    .groupby("decile")
    .agg(n=("y", "size"), mean_pred=("p", "mean"), observed=("y", "mean"))
)
reliability.to_csv(TAB / "late_delivery_reliability_test.csv")
reliability

## 6. Segment error analysis on the test window (champion)
Recall/precision at budget use the single test-window top-10 % threshold, so a segment's
flagged share shows where alerts would concentrate. AUCs are NaN with < 10 positives.

In [ ]:
test = parts["test"].assign(p=p_test)
t_test = np.quantile(p_test, 1 - BUDGET)
test["flag"] = test["p"] >= t_test


def segment(frame, key, top=None):
    out = []
    for value, g in frame.groupby(key, observed=True):
        yy, pp = g[LABEL].astype(int).to_numpy(), g["p"].to_numpy()
        ok = 10 <= yy.sum() < len(yy)
        out.append(
            {
                key: str(value),
                "n": len(g),
                "late_rate": yy.mean(),
                "mean_pred": pp.mean(),
                "pr_auc": average_precision_score(yy, pp) if ok else np.nan,
                "roc_auc": roc_auc_score(yy, pp) if ok else np.nan,
                "brier": brier_score_loss(yy, pp),
                "flagged_share": g["flag"].mean(),
                "recall_at_budget": (g["flag"] & (yy == 1)).sum() / max(1, yy.sum()),
                "precision_at_budget": (g["flag"] & (yy == 1)).sum() / max(1, g["flag"].sum()),
            }
        )
    t = pd.DataFrame(out)
    return t.sort_values("n", ascending=False).head(top) if top else t  # buckets keep bin order


segments = {
    "customer_state": segment(test, "customer_state", 10),
    "product_category": segment(test, "product_category", 10),
    "distance_bucket": segment(test, "distance_bucket"),
    "freight_ratio_bucket": segment(test, "freight_ratio_bucket"),
}
for name, t in segments.items():
    t.to_csv(TAB / f"late_delivery_test_by_{name}.csv", index=False)
print("test overall:", at_budget(y["test"], p_test, t_test))
segments["customer_state"]

In [ ]:
segments["product_category"]

In [ ]:
segments["distance_bucket"]

In [ ]:
segments["freight_ratio_bucket"]

## 7. SHAP (TreeSHAP) on a test sample with the champion
LightGBM's native `pred_contrib` (exact TreeSHAP, log-odds units) on the champion's own feature
step; the `shap` package is only used for the beeswarm plot. Categoricals are plotted by
category code (colour is not meaningful for them).

In [ ]:
pipe = models["lightgbm"].unwrap_python_model().pipeline
sample = X["test"].sample(n=min(3000, len(X["test"])), random_state=42)
F = pipe.named_steps["features"].transform(sample)
contrib = pipe.named_steps["model"].predict(F, pred_contrib=True)
phi, base = contrib[:, :-1], contrib[:, -1]
p_sample = np.asarray(models["lightgbm"].predict(sample), dtype=float)
assert np.allclose(1 / (1 + np.exp(-(phi.sum(1) + base))), p_sample, atol=1e-6)
F_plot = F.copy()
for col in CATEGORICAL:
    F_plot[col] = F_plot[col].cat.codes.replace(-1, np.nan)
F_plot = F_plot.astype(float)
numeric = [c for c in F.columns if c not in CATEGORICAL]
importance = pd.DataFrame(
    {
        "mean_abs_shap": np.abs(phi).mean(0),
        "spearman_value_vs_shap": [
            pd.Series(F_plot[c]).corr(pd.Series(phi[:, i], index=F_plot.index), method="spearman")
            if c in numeric
            else np.nan
            for i, c in enumerate(F.columns)
        ],
    },
    index=F.columns,
).sort_values("mean_abs_shap", ascending=False)
importance.to_csv(TAB / "late_delivery_shap_importance.csv")
importance

In [ ]:
cat_effect = {}
for col in CATEGORICAL:
    i = list(F.columns).index(col)
    s = pd.Series(phi[:, i]).groupby(F[col].astype(str).to_numpy()).agg(["mean", "size"])
    s = s[s["size"] >= 30].sort_values("mean")
    cat_effect[col] = {
        "lowest": s.head(3)["mean"].round(3).to_dict(),
        "highest": s.tail(3)["mean"].round(3).to_dict(),
    }
for col, eff in cat_effect.items():
    print(f"{col}: lowest mean SHAP {eff['lowest']} | highest {eff['highest']}")

In [ ]:
shap.summary_plot(phi, F_plot, max_display=12, show=False, plot_size=(9, 6))
plt.title(f"late_delivery v{version.version}: TreeSHAP on {len(sample)} test orders (log-odds)")
plt.tight_layout()
plt.savefig(FIG / "late_delivery_shap_summary.png", dpi=120)
plt.show()